# pMTG Clustering With an INR Residualization Parameter

Set `RESIDUALIZE_FC_FOR_INR` in the parameter cell to run the same clustering workflow with or without INR in the FC residualization model. The repeated-run agreement, bootstrap stability, and two-/four-cluster PCA figures are shared.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import seaborn as sns
sns.set_palette('pastel')
sns.set_style('whitegrid')
import nibabel as nib
import os
from sklearn.metrics import pairwise_distances
import statsmodels.api as sm
from statsmodels.formula.api import ols
from sklearn.metrics import silhouette_score, pairwise_distances, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score
from networkx.algorithms.community import louvain_communities
import networkx as nx
from infomap import Infomap
from scipy.stats import mode

from sklearn.linear_model import LinearRegression
from statsmodels.stats.multitest import multipletests

import warnings
warnings.filterwarnings('ignore')


# Paths

In [ ]:
# Notebook-local analysis helpers.
STANDARD_FC_COVARIATES = [
    'demo_sex_v2',
    'interview_age',
    'site_id_l',
    'ehi1b',
    'mean_fd_0.20',
]

DEFAULT_CATEGORICAL_COVARIATES = {
    'demo_sex_v2',
    'site_id_l',
    'ehi1b'
}


def standardize_subject_id(subject_ids):
    return (
        subject_ids.astype(str)
        .str.replace('_', '', regex=False)
        .str.replace('^sub-', '', regex=True)
    )


def load_motion_qa(motion_qa_path, motion_column='mean_fd_0.20'):
    motion_qa = pd.read_csv(motion_qa_path)
    motion_qa = motion_qa[['src_subject_id', motion_column]].copy()
    motion_qa['src_subject_id'] = standardize_subject_id(motion_qa['src_subject_id'])
    motion_qa = motion_qa.drop_duplicates(subset=['src_subject_id'])
    motion_qa[motion_column] = pd.to_numeric(motion_qa[motion_column], errors='coerce')
    return motion_qa


def merge_motion_qa(df, motion_qa_path, motion_column='mean_fd_0.20', how='left'):
    motion_qa = load_motion_qa(motion_qa_path, motion_column=motion_column)

    merged = df.copy()
    merged['src_subject_id'] = standardize_subject_id(merged['src_subject_id'])
    if motion_column in merged.columns:
        existing_motion = merged.groupby('src_subject_id')[motion_column].first()
        merged = merged.drop(columns=[motion_column])
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
        merged[motion_column] = merged[motion_column].fillna(
            merged['src_subject_id'].map(existing_motion)
        )
    else:
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
    return merged


def complete_covariate_mask(df, covariates, categorical_covariates=DEFAULT_CATEGORICAL_COVARIATES):
    categorical_covariates = set(categorical_covariates)
    complete = pd.Series(True, index=df.index)

    for covariate in covariates:
        is_categorical = df[covariate].dtype == 'object' or covariate in categorical_covariates
        if is_categorical:
            complete &= df[covariate].notna()
        else:
            complete &= pd.to_numeric(df[covariate], errors='coerce').notna()

    return complete

def encode_regression_covariates(df, covariates, categorical_covariates=DEFAULT_CATEGORICAL_COVARIATES):
    encoded_parts = []
    categorical_covariates = set(categorical_covariates)

    for covariate in covariates:
        if df[covariate].dtype == 'object' or covariate in categorical_covariates:
            encoded_parts.append(
                pd.get_dummies(
                    df[covariate],
                    prefix=covariate,
                    drop_first=True,
                    dtype=float,
                )
            )
        else:
            encoded_parts.append(
                pd.to_numeric(df[covariate], errors='coerce').to_frame(covariate)
            )

    if not encoded_parts:
        return pd.DataFrame(index=df.index)
    return pd.concat(encoded_parts, axis=1)


def residualize_fc_profiles(df, fc_columns, covariates=STANDARD_FC_COVARIATES):
    df = df.copy()
    covariate_complete = complete_covariate_mask(df, covariates)

    validity_groups = {}
    for column in fc_columns:
        valid_idx = df[column].notnull() & covariate_complete
        key = valid_idx.to_numpy(dtype=np.bool_).tobytes()
        if key not in validity_groups:
            validity_groups[key] = (valid_idx, [])
        validity_groups[key][1].append(column)

    residual_frames = []
    for valid_idx, columns in validity_groups.values():
        output_columns = [column + '_resid' for column in columns]
        residuals = pd.DataFrame(np.nan, index=df.index, columns=output_columns)
        if valid_idx.sum() > 0:
            observed = df.loc[valid_idx, columns].to_numpy()
            covariate_matrix = encode_regression_covariates(
                df.loc[valid_idx],
                covariates,
            )
            if covariate_matrix.shape[1] == 0:
                predicted = np.tile(observed.mean(axis=0), (len(observed), 1))
            else:
                model = LinearRegression()
                model.fit(covariate_matrix, observed)
                predicted = model.predict(covariate_matrix)
            residuals.loc[valid_idx, output_columns] = observed - predicted
        residual_frames.append(residuals)

    if residual_frames:
        df = pd.concat([df, *residual_frames], axis=1)

    return df




To ABCD tabulated data directory (ABCD_DATA_DIR), the functional connectivity data (FC_PATH; pMTG to each of 14 large-scale resting-state networks for the ABCD participants), and the output folder (OUTPUT_DIR).

In [ ]:
ABCD_DATA_DIR = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Data/abcd-data-release-5.1/core'
FC_PATH = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/wrangled_pMTG_FC_data_midb61_meanFC.csv'
OUTPUT_DIR = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final'
MOTION_QA_PATH = os.path.join(OUTPUT_DIR, 'motion_QA_results.csv')

# This flag controls whether INR is included with the standard FC residualization covariates.
RESIDUALIZE_FC_FOR_INR = False
INR_VARIABLE = 'inr'
INR_RESIDUALIZATION_COVARIATE = 'inr'
CLUSTERING_OUTPUT_LABEL = 'motion_inr_resid' if RESIDUALIZE_FC_FOR_INR else 'motion_resid'

# Full analysis defaults. Reduce these only for development or smoke testing.
N_CLUSTERING_RUNS = 1001
N_CLUSTER_BOOTSTRAPS = 5000
CLUSTER_BOOTSTRAP_RANDOM_STATE = 42


# Utilities

Functions to assist with unsupervised clustering based on pMTG-network FC data and plotting the solutions. 

In [ ]:

def add_infomap_community_assignments(df, community_dict, i=2):
    """
    Add Infomap community assignments to the DataFrame.

    Parameters:
    - df: DataFrame with 'src_subject_id' column
    - community_dict: dict {community_id: list of src_subject_ids}
    - i: integer to append to column name for uniqueness. Default = 2 (since the most stable solution has 2 communities)

    Returns:
    - df with new column 'infomap_community'
    """
    # Create a mapping from subject ID to community ID
    subject_to_community = {}
    for community_id, subjects in community_dict.items():
        for subject in subjects:
            subject_to_community[subject] = community_id

    # Map the communities to the DataFrame
    df[f'infomap_community_{i}'] = df['src_subject_id'].map(subject_to_community)
    
    return df

def radar_plot(df, comm_col, colors=['blue','orange','green','red','purple'],Pos=True):
    """
    Create radar plots for left and right hemisphere data across communities,
    with a consistent scale across all subplots.

    Parameters:
    - df: DataFrame with columns as features and rows as subjects
    - comm_col: Name of the column indicating community assignment
    """
    # Identify left and right hemisphere columns
    # Create labels 
    left_cols = [col for col in df.columns if '_L_' in col and col.endswith('_resid') and 'full' not in col]
    right_cols = [col for col in df.columns if '_R_' in col and col.endswith('_resid') and 'full' not in col]
    labels_left = [col.split('_L_')[0] for col in left_cols]
    labels_right = [col.split('_R_')[0] for col in right_cols]
    labels_left = [label.split('_')[-1] + label.split('_')[0] for label in labels_left]
    labels_right = [label.split('_')[-1] + label.split('_')[0] for label in labels_right]
    labels_left = [label.replace('left', 'L_') for label in labels_left]
    labels_left = [label.replace('right', 'R_') for label in labels_left]
    labels_right = [label.replace('left', 'L_') for label in labels_right]
    labels_right = [label.replace('right', 'R_') for label in labels_right]

    # Compute angles
    angles_left = np.linspace(0, 2 * np.pi, len(left_cols), endpoint=False).tolist()
    angles_left += angles_left[:1]
    angles_right = np.linspace(0, 2 * np.pi, len(right_cols), endpoint=False).tolist()
    angles_right += angles_right[:1]

    # Function to compute mean radar profile per community
    def get_mean_profile(df, columns, community_id):
        profile = df[df[comm_col] == community_id][columns].mean().tolist()
        profile += profile[:1]  # close the radar
        if Pos:
            profile = [max(0, val) for val in profile]
        return profile

    # Compute global min/max for scaling
    all_profiles_left = []
    all_profiles_right = []
    communities = df[comm_col].unique()

    for comm in communities:
        all_profiles_left.append(get_mean_profile(df, left_cols, comm)[:-1])
        all_profiles_right.append(get_mean_profile(df, right_cols, comm)[:-1])
    if Pos:
        global_min = 0
    else:
        global_min = min(np.min(all_profiles_left), np.min(all_profiles_right))
    global_max = max(np.max(all_profiles_left), np.max(all_profiles_right))

    # Plot
    fig, axs = plt.subplots(len(communities), 2, subplot_kw=dict(polar=True), figsize=(10, len(communities)*4))

    # Order communities by mean DMN_left_L_fz_resid
    dmn_left_col = 'DMN_left_L_fz_resid'
    if dmn_left_col in df.columns:
        dmn_means = df.groupby(comm_col)[dmn_left_col].mean()
        ordered_communities = dmn_means.sort_values().index.tolist()
    else:
        ordered_communities = sorted(communities, key=lambda x: int(x))
    communities = [comm for comm in ordered_communities if comm in df[comm_col].unique()]
    
    for i, comm in enumerate(communities):
        # Left hemisphere
        profile_left = get_mean_profile(df, left_cols, comm)
        ax_left = axs[i, 0]
        ax_left.plot(angles_left, profile_left, linewidth=2, color=colors[i % len(colors)])
        ax_left.fill(angles_left, profile_left, alpha=0.3, color=colors[i % len(colors)])
        ax_left.set_title(f'Community {comm+1} – Left Hemisphere', fontsize=12, weight='bold')
        ax_left.set_xticks(angles_left[:-1])
        ax_left.set_xticklabels(labels_left, fontsize=7)
        for label in ax_left.get_xticklabels():
            label.set_rotation(45)
        ax_left.set_ylim(global_min, global_max)

        # Right hemisphere
        profile_right = get_mean_profile(df, right_cols, comm)
        ax_right = axs[i, 1]
        ax_right.plot(angles_right, profile_right, linewidth=2, color=colors[i % len(colors)])
        ax_right.fill(angles_right, profile_right, alpha=0.3, color=colors[i % len(colors)])
        ax_right.set_title(f'Community {comm+1} – Right Hemisphere', fontsize=12, weight='bold')
        ax_right.set_xticks(angles_right[:-1])
        ax_right.set_xticklabels(labels_right, fontsize=7)
        for label in ax_right.get_xticklabels():
            label.set_rotation(45)
        ax_right.set_ylim(global_min, global_max)

    plt.tight_layout()
    plt.show()

def silhouette_im(data, community_dict, node_order):
    """
    Compute silhouette score given data and Infomap or Louvain community assignments.

    Parameters:
    - data: DataFrame of shape (n_subjects, n_features), indexed by subject ID
    - community_dict: dict {community_id: list of subject_ids}
    - node_order: list of subject_ids, order must match rows in `data`

    Returns:
    - silhouette score and number of communities
    """
    # Build label vector from community_dict
    subject_to_label = {
        subj: label for label, subjects in community_dict.items() for subj in subjects
    }

    # Extract data in correct order
    labels = []
    valid_subjects = []
    for subj in node_order:
        if subj in subject_to_label:
            labels.append(subject_to_label[subj])
            valid_subjects.append(subj)

    X = data.loc[valid_subjects].values
    distance_matrix = pairwise_distances(X, metric='correlation') 
    score = silhouette_score(distance_matrix, labels, metric='precomputed')
    print(f"Silhouette index solution with {len(set(labels))} communities: {score:.4f}")
    return score

def davies_bouldin_im(data, community_dict, node_order):
    """
    Compute Davies-Bouldin score given data and Infomap community assignments.

    Parameters:
    - data: DataFrame of shape (n_subjects, n_features), indexed by subject ID
    - community_dict: dict {community_id: list of subject_ids}
    - node_order: list of subject_ids, order must match rows in `data`

    Returns:
    - Davies-Bouldin score
    """
    # Build label vector from community_dict
    subject_to_label = {
        subj: label for label, subjects in community_dict.items() for subj in subjects
    }

    # Extract data in correct order
    labels = []
    valid_subjects = []
    for subj in node_order:
        if subj in subject_to_label:
            labels.append(subject_to_label[subj])
            valid_subjects.append(subj)

    if len(set(labels)) < 2:
        print("Only one community found, Davies-Bouldin index set to 10.")
        db_score = 10
    else:
        X = data.loc[valid_subjects].values
        db_score = davies_bouldin_score(X, labels)
        print(f"Davies-Bouldin index for solution with {len(set(labels))} communities: {db_score:.4f}")
    return db_score


## Infomap

In [ ]:
def find_networks(df, src_subject_ids, thresholds=[0.1, 0.2, 0.3, 0.35, 0.4, 0.43]):
    """
    Constructs a subject similarity graph using Pearson correlation of FC profiles,
    applies thresholding, and detects communities using Infomap.

    Parameters:
    - df: DataFrame of shape (n_subjects, n_features) containing FC features per subject
    - src_subject_ids: array-like of subject IDs, one per row in `df`
    - thresholds: list of percentile thresholds for edge inclusion (0–1 scale)

    Returns:
    - all_communities: dict mapping each threshold -> {community_id: list of subject IDs}
    - all_graphs: dict mapping each threshold -> networkx Graph instance
    """
    # Convert the FC matrix to numeric values and verify that it matches the subject vector.
    data = df.to_numpy(dtype=float)
    src_subject_ids = np.asarray(src_subject_ids)

    all_communities = {}  # Stores communities per threshold
    all_graphs = {}       # Stores graphs per threshold

    # Build mapping between numeric index and subject ID
    idx_to_subj = dict(enumerate(src_subject_ids))
    subj_to_idx = {v: k for k, v in idx_to_subj.items()}

    # Compute Pearson correlation matrix across subjects
    similarity_matrix = np.corrcoef(data)
    print(f"Similarity matrix shape: {similarity_matrix.shape}")
    n_subjects = similarity_matrix.shape[0]

    # Iterate over each threshold
    for threshold in thresholds:
        # Initialize graph with all subjects as nodes
        G = nx.Graph()
        G.add_nodes_from(src_subject_ids)

        # Extract upper triangle of similarity matrix (excluding diagonal)
        upper_tri = similarity_matrix[np.triu_indices(n_subjects, k=1)]

        # Determine correlation cutoff corresponding to given threshold percentile
        cutoff = np.percentile(upper_tri, 100 - threshold * 100)
        print(f"[Threshold {threshold:.3f}] Correlation cutoff: {cutoff:.4f}")

        # Add edges between subject pairs exceeding the correlation threshold
        for i in range(n_subjects):
            for j in range(i + 1, n_subjects):
                if similarity_matrix[i, j] > cutoff:
                    G.add_edge(src_subject_ids[i], src_subject_ids[j], weight=similarity_matrix[i, j])

        # Guard against thresholds that produce empty graphs

        # Initialize Infomap and add edges using integer node IDs
        im = Infomap()
        for u, v in G.edges():
            im.add_link(subj_to_idx[u], subj_to_idx[v])

        # Run Infomap community detection (with fixed seed for reproducibility)
        im.run(seed=42)

        # Extract communities and map back to original subject IDs
        communities = {}
        for node in im.nodes:
            subj_id = idx_to_subj[node.node_id]
            communities.setdefault(node.module_id, []).append(subj_id)

        # Store results for current threshold
        all_communities[threshold] = communities
        all_graphs[threshold] = G

        print(f"[Threshold {threshold:.3f}] Found {len(communities)} communities.")

        # Optional: Reorder communities by mean DMN activation, if column is present
        if 'DMN_left_L_fz_resid' in df.columns:
            # Map each subject ID to its DMN value
            dmn_col = df['DMN_left_L_fz_resid']
            subj_to_dmn = dict(zip(src_subject_ids, dmn_col))

            # Compute mean DMN value for each community
            community_means = {
                comm_id: np.mean([subj_to_dmn[subj] for subj in members])
                for comm_id, members in communities.items()
            }

            # Sort communities by increasing mean DMN value
            ordered = sorted(community_means.items(), key=lambda x: x[1])

            # Reassign community IDs based on sorted order (0, 1, 2, ...)
            new_communities = {
                i: communities[comm_id]
                for i, (comm_id, _) in enumerate(ordered)
            }

            # Replace original community structure with reordered version
            all_communities[threshold] = new_communities

    return all_communities, all_graphs


In [ ]:
# Load the wrangled FC table, merge motion QA, and prepare the FC columns used for clustering.
df = pd.read_csv(FC_PATH)
print(f'Rows loaded from wrangled FC table: {len(df)}')

df = merge_motion_qa(df, MOTION_QA_PATH, how='left')
print(f'Rows after motion QA merge: {len(df)}')

# Drop generated full-network Fisher-z FC columns before selecting clustering features.
full_fc_cols = [column for column in df.columns if '_fz' in str(column) and '_full' in str(column)]
if full_fc_cols:
    df = df.drop(columns=full_fc_cols)
print(f'Dropped {len(full_fc_cols)} generated full-network Fisher-z FC columns from the clustering dataframe.')


def subject_edge_cutoff(data, threshold):
    # Report the cutoff from the same subject-by-subject similarity matrix used for graph construction.
    complete_data = data.dropna()
    similarity_matrix = np.corrcoef(complete_data)
    upper_triangle = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]
    return np.percentile(upper_triangle, 100 - threshold * 100)

# Define the network labels expected in the raw pMTG FC column names.
network_labels = {
    'DMN': 1, 'VAN': 7, 'Aud': 12, 'CO': 9, 'PMN': 15,
    'DAN': 5, 'FP': 3, 'PON': 16, 'Sal': 8,
    'SMd': 10, 'SMl': 11, 'Vis': 2, 'Tpole': 13, 'MTL': 14,
}

# Use only network-specific Fisher-z FC columns; full-network columns were removed above.
raw_fc_columns = [col for col in df.columns if col.endswith('_fz') and '_full' not in col]

expected_resid_columns = [f'{col}_resid' for col in raw_fc_columns]
existing_resid_columns = [col for col in expected_resid_columns if col in df.columns]
cluster_covariates = list(STANDARD_FC_COVARIATES)

if RESIDUALIZE_FC_FOR_INR:
    required_inr_cols = [INR_VARIABLE, 'inr_missing', 'poverty_line_2017']
    missing_inr_cols = [col for col in required_inr_cols if col not in df.columns]
    cluster_covariates = cluster_covariates + [INR_RESIDUALIZATION_COVARIATE]

print('Clustering FC covariates:', cluster_covariates)
for covariate in cluster_covariates:
    missing = df[covariate].isna().sum()
    print(f'Missing {covariate} values before FC residualization: {missing}')

if RESIDUALIZE_FC_FOR_INR:
    if existing_resid_columns:
        df = df.drop(columns=existing_resid_columns)
        print(
            'Dropped existing no-INR residualized FC columns before INR residualization: '
            f'{len(existing_resid_columns)}'
        )
    print(f'Missing raw INR values left as NaN for FC residualization: {df[INR_VARIABLE].isna().sum()}')
    df = residualize_fc_profiles(df, raw_fc_columns, covariates=cluster_covariates)
else:
    missing_resid_columns = [col for col in expected_resid_columns if col not in df.columns]
    if missing_resid_columns:
        print('Residualized FC columns were not found in the wrangled table; recomputing them here.')
        df = residualize_fc_profiles(df, raw_fc_columns, covariates=cluster_covariates)
    else:
        print(f'Using existing residualized FC columns from the wrangled table: {len(existing_resid_columns)}')

# Keep the expected residualized FC column names so downstream clustering cells match the reference notebook.
resid_columns = expected_resid_columns
cluster_resid_columns = resid_columns
duplicate_resid_columns = [col for col in resid_columns if list(df.columns).count(col) > 1]

n_before_complete = len(df)
cluster_complete_mask = df[resid_columns].notna().all(axis=1)
n_complete = int(cluster_complete_mask.sum())
n_excluded = int((~cluster_complete_mask).sum())
print(f'Rows retained with complete clustering FC profiles: {n_complete}/{n_before_complete}')
print(f'Rows excluded for incomplete clustering FC profiles: {n_excluded}')
df = df.loc[cluster_complete_mask].copy()
df.reset_index(drop=True, inplace=True)

if 'rel_family_id' in df.columns:
    duplicate_family_rows = df['rel_family_id'].dropna().duplicated().sum()
    print(f'Rows from non-missing families already represented after clustering filtering: {duplicate_family_rows}')
if 'good_frames_0.20' in df.columns:
    good_frames = pd.to_numeric(df['good_frames_0.20'], errors='coerce')
    print(f'Rows below 600 good frames after clustering filtering: {(good_frames < 600).sum()}')

fc_profile_dict = {col: i for i, col in enumerate(resid_columns)}
print('FC profile dictionary:', fc_profile_dict)

fc_profile_left_columns = [col for col in resid_columns if '_L_' in col]
fc_profile_right_columns = [col for col in resid_columns if '_R_' in col]
print('Left residualized FC profile columns:', fc_profile_left_columns)
print('Right residualized FC profile columns:', fc_profile_right_columns)
print(df)


### Run Infomap

In [ ]:
src_subject_ids = df['src_subject_id'].values
thresholds = [i/100 for i in range(1,51)]
communities, graphs = find_networks(df[resid_columns], src_subject_ids, thresholds)


Calculating silhouette scores

In [ ]:
s_scores = []
for threshold in thresholds:
    comms = communities[threshold]
    if len(comms) < 2:
        print(f"Threshold {threshold:.3f}: No communities found.")
        s_scores.append(None)
        continue
    # make new df with just src_subject_id and the residualized FC columns
    df_temp = df[['src_subject_id'] + resid_columns].copy()
    df_temp.set_index('src_subject_id', inplace=True)
    score = silhouette_im(df_temp, comms, src_subject_ids)
    s_scores.append(score)
    print(f"Threshold {threshold:.3f}: Silhouette score = {score:.4f}")

# Plot
plt.figure(figsize=(10, 6))
valid_thresholds = [t for t, s in zip(thresholds, s_scores) if s is not None]
valid_scores = [s for s in s_scores if s is not None]
plt.plot(valid_thresholds, valid_scores, marker='o', color='black')
plt.xlabel('Threshold')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score by Threshold')
plt.grid()
plt.show()


In [ ]:
# Calculate Davies-Bouldin score for each threshold
db_scores = []
for threshold in thresholds:
    comms = communities[threshold]
    df_temp = df[['src_subject_id'] + resid_columns].copy()
    df_temp.set_index('src_subject_id', inplace=True)
    db_score = davies_bouldin_im(df_temp, comms, src_subject_ids)
    db_scores.append(db_score)
    print(f"Threshold {threshold:.3f}: Davies-Bouldin index = {db_score:.4f}")
    

# Plot Davies-Bouldin scores
plt.figure(figsize=(10, 6))
good_db_scores = [d for d in db_scores if d != 10]  # Exclude scores set to 10
good_thresholds = [thresholds[i] for i, d in enumerate(db_scores) if d != 10]
plt.plot(good_thresholds, good_db_scores, marker='o', color='black')
plt.xlabel('Threshold')
plt.ylabel('Davies-Bouldin Index')
plt.title('Davies-Bouldin Index by Threshold')
plt.grid()
plt.show()


In [ ]:
# find threshold with lowest DBI score at silhouette score > 0.245
thresholds = np.array(thresholds)
db_scores = np.array(db_scores)
s_scores = np.array(s_scores, dtype=object)

# Only consider thresholds with a usable silhouette score.
valid_indices = np.array([i for i, s in enumerate(s_scores) if s is not None and s > 0.20])

min_db_index = valid_indices[np.argmin(db_scores[valid_indices])]
best_threshold = thresholds[min_db_index]
print(f'Best threshold with silhouette score > 0.20: {best_threshold:.3f}')
print(f'Associated Pearson correlation cutoff: {subject_edge_cutoff(df[resid_columns], best_threshold):.4f}')
print(f'Minimum Davies-Bouldin score at this threshold: {db_scores[min_db_index]:.4f}')
print(f'Silhouette score at this threshold: {s_scores[min_db_index]:.4f}')

threshold = best_threshold
community_dict = communities[threshold]
print(f'Number of communities at threshold {threshold}: {len(community_dict)}')
for community_id, subjects in community_dict.items():
    print(f'Community {community_id}: {len(subjects)} subjects')

# Add Infomap community assignments without resetting the dataframe index.
df = add_infomap_community_assignments(df, community_dict, i=2)
print(df.head())


In [ ]:
radar_plot(df, 'infomap_community_2')


In [ ]:
# add best 3 and 4 network Infomap solutions based on lowest DBI and high silhouette score
# there should be only one best solution for 3 and 4 communities
# 4/27.2026 - changed silhouette score threshold to 0.10 to find valid solutions for 3 and 4 communities using mean FC rather than mean TS FC data
thresholds = np.array(thresholds)
db_scores = np.array(db_scores)
s_scores = np.array(s_scores, dtype=object)
valid_indices_3 = np.array([i for i, s in enumerate(s_scores) if s is not None and s > 0.10 and len(communities[thresholds[i]]) == 3])
valid_indices_4 = np.array([i for i, s in enumerate(s_scores) if s is not None and s > 0.10 and len(communities[thresholds[i]]) == 4])
if valid_indices_3.size == 0:
    print("No valid thresholds found with silhouette score > 0.10 and 3 communities")
else:
    min_db_index_3 = valid_indices_3[np.argmin(db_scores[valid_indices_3])]
    best_threshold_3 = thresholds[min_db_index_3]
    print(f"Best threshold with silhouette score > 0.10 and 3 communities: {best_threshold_3:.3f}")
    print(f"Associated Pearson correlation cutoff: {subject_edge_cutoff(df[resid_columns], best_threshold_3):.4f}")
    print(f"Minimum Davies-Bouldin score at this threshold: {db_scores[min_db_index_3]:.4f}")
    print(f"Silhouette score at this threshold: {s_scores[min_db_index_3]:.4f}")
    community_dict_3 = communities[best_threshold_3]
    df = add_infomap_community_assignments(df, community_dict_3, i=3)
    radar_plot(df, 'infomap_community_3')
if valid_indices_4.size == 0:
    print("No valid thresholds found with silhouette score > 0.10 and 4 communities")
else:
    min_db_index_4 = valid_indices_4[np.argmin(db_scores[valid_indices_4])]
    best_threshold_4 = thresholds[min_db_index_4]
    print(f"Best threshold with silhouette score > 0.10 and 4 communities: {best_threshold_4:.3f}")
    print(f"Associated Pearson correlation cutoff: {subject_edge_cutoff(df[resid_columns], best_threshold_4):.4f}")
    print(f"Minimum Davies-Bouldin score at this threshold: {db_scores[min_db_index_4]:.4f}")
    print(f"Silhouette score at this threshold: {s_scores[min_db_index_4]:.4f}")
    community_dict_4 = communities[best_threshold_4]
    df = add_infomap_community_assignments(df, community_dict_4, i=4)
    radar_plot(df, 'infomap_community_4')


# K-means Clustering

Clustering individuals using the KMeans algorithm with Euclidean distance as the distance metric.

In [ ]:
ns = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
kmeans_WCSS = {}

# KMeans uses Euclidean distance internally; silhouette scores below use correlation distance.
resid_columns = cluster_resid_columns
print(f'Rows used for KMeans clustering: {len(df)}')
for n in ns:
    kmeans = KMeans(n_clusters=n, random_state=42)
    kmeans.fit(df[resid_columns])

    df[f'kmeans_{n}_labels'] = kmeans.labels_
    kmeans_WCSS[n] = kmeans.inertia_

    print(f'KMeans with {n} clusters done.')


In [ ]:
# plot elbow plot for kmeans WCSS
plt.figure(figsize=(8, 5))
plt.plot(list(kmeans_WCSS.keys()), list(kmeans_WCSS.values()), marker='o', color='black')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.title('Elbow Method for Optimal k in KMeans')
plt.xticks(ns)
plt.grid()
plt.show()


In [ ]:
# calculate the db score for each kmeans clustering
kmeans_db_scores = {}
for n in ns:
    if f'kmeans_{n}_labels' in df.columns:
        db_score = davies_bouldin_score(df[resid_columns], df[f'kmeans_{n}_labels'])
        kmeans_db_scores[n] = db_score
        print(f"KMeans with {n} clusters: Davies-Bouldin index = {db_score:.4f}")
# plot db scores for kmeans clustering
plt.figure(figsize=(8, 5))
plt.plot(list(kmeans_db_scores.keys()), list(kmeans_db_scores.values()), marker='o', color='black')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Davies-Bouldin Index')
plt.title('Davies-Bouldin Index for KMeans Clustering')
plt.xticks(ns)
plt.grid()
plt.show()


In [ ]:
# calculate silhouette scores for each kmeans clustering
kmeans_silhouette_scores = {}
for n in ns:
    if f'kmeans_{n}_labels' in df.columns:
        distance_matrix = pairwise_distances(df[resid_columns], metric='correlation')
        score = silhouette_score(distance_matrix, df[f'kmeans_{n}_labels'], metric='precomputed')
        kmeans_silhouette_scores[n] = score
        print(f'Silhouette score for KMeans with {n} clusters: {score:.4f}')
    else:
        print(f'Column kmeans_{n}_labels not found in DataFrame.')


In [ ]:
# plot kmeans silhouette scores
plt.figure(figsize=(8, 5))
plt.plot(list(kmeans_silhouette_scores.keys()), list(kmeans_silhouette_scores.values()), marker='o', color='black')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for K-Means Clustering')
plt.xticks(ns)
plt.grid()
plt.show()


In [ ]:
# plot silhouette scores for kmeans clustering
plt.figure(figsize=(8, 5))
plt.plot(list(kmeans_silhouette_scores.keys()), list(kmeans_silhouette_scores.values()), marker='o', color='black')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for K-Means Clustering')
plt.xticks(ns)
plt.grid()
plt.show()


In [ ]:
radar_plot(df, 'kmeans_2_labels')
radar_plot(df, 'kmeans_3_labels')
radar_plot(df, 'kmeans_4_labels')


In [ ]:
# Run k-means repeatedly and save every subject assignment.
# The consensus solution is the modal label across runs for each participant.
k = 2
n_runs = N_CLUSTERING_RUNS
all_labels = np.zeros((df.shape[0], n_runs))
for run in range(n_runs):
    kmeans = KMeans(n_clusters=k, random_state=run)
    labels = kmeans.fit_predict(df[resid_columns])

    # Sort labels by mean DMN_left_L_fz_resid value so labels are consistent across runs.
    dmn_col = 'DMN_left_L_fz_resid'
    if dmn_col in df.columns:
        dmn_means = {}
        for label in np.unique(labels):
            dmn_means[label] = df.loc[labels == label, dmn_col].mean()
        sorted_labels = sorted(dmn_means, key=dmn_means.get)
        label_mapping = {old_label: new_label for new_label, old_label in enumerate(sorted_labels)}
        labels = np.array([label_mapping[label] for label in labels])

    df['kmeans_2_run_' + str(run+1)] = labels
    all_labels[:, run] = labels
    if (run + 1) % 100 == 0:
        print(f'KMeans run {run + 1}/{n_runs} done.')

from scipy.stats import mode
consensus_labels, _ = mode(all_labels, axis=1)
df['kmeans_2_consensus'] = consensus_labels.flatten()
print(df['kmeans_2_consensus'].value_counts())


# Louvain

In [ ]:
def find_networks_louvain(data, src_subject_ids, thresholds=[0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45], res=1, s=42):
    """
    Constructs a subject similarity graph using Pearson correlation of FC profiles,
    applies thresholding, and detects communities using Louvain.

    Parameters:
    - data: DataFrame of shape (n_subjects, n_features)
    - src_subject_ids: array-like of subject IDs, one per row in `data`
    - thresholds: list of percentile thresholds for edge inclusion

    Returns:
    - all_communities: dict mapping threshold -> {community_id: list of src_subject_id}
    - all_graphs: dict mapping threshold -> networkx Graph
    """

    data = data.to_numpy(dtype=float)
    src_subject_ids = np.asarray(src_subject_ids)

    all_communities = {}
    all_graphs = {}

    similarity_matrix = np.corrcoef(data)
    n_subjects = similarity_matrix.shape[0]

    for threshold in thresholds:
        G = nx.Graph()
        G.add_nodes_from(src_subject_ids)

        # Compute threshold cutoff
        upper_tri = similarity_matrix[np.triu_indices(n_subjects, k=1)]
        cutoff = np.percentile(upper_tri, 100 - threshold * 100)
        print(f"[Threshold {threshold:.3f}] Correlation cutoff: {cutoff:.4f}")

        # Add edges above threshold
        for i in range(n_subjects):
            for j in range(i + 1, n_subjects):
                if similarity_matrix[i, j] > cutoff:
                    G.add_edge(src_subject_ids[i], src_subject_ids[j], weight=similarity_matrix[i, j])


        # Run Louvain community detection
        partition = louvain_communities(G, resolution=res, seed=s)

        # Map detected communities back to subject IDs
        communities = {i: list(comm) for i, comm in enumerate(partition)}
        all_communities[threshold] = communities
        all_graphs[threshold] = G

        print(f"[Threshold {threshold:.3f}] Found {len(communities)} communities.")

        # re-number communities based on mean DMN_left_L_fz_resid
        if 'DMN_left_L_fz_resid' in df.columns:
            # Map each subject ID to its DMN value
            dmn_col = df['DMN_left_L_fz_resid']
            subj_to_dmn = dict(zip(src_subject_ids, dmn_col))

            # Compute mean DMN value for each community
            community_means = {
                comm_id: np.mean([subj_to_dmn[subj] for subj in members])
                for comm_id, members in communities.items()
            }

            # Sort communities by increasing mean DMN value
            ordered = sorted(community_means.items(), key=lambda x: x[1])

            # Reassign community IDs based on sorted order 
            new_communities = {
                i: communities[comm_id]
                for i, (comm_id, _) in enumerate(ordered)
            }

            # Replace original community structure with reordered version
            all_communities[threshold] = new_communities

    return all_communities, all_graphs

# Run Louvain community detection
ress = [0, 0.25, 0.5, 0.75, 1, 1.25, 1.5]
src_subject_ids = df['src_subject_id'].values
# First, find the maximum silhouette score for each resolution
thresholds = [i/20 for i in range(1, 10)]  # 0.05 to 0.45 in steps of 0.05
max_silhouette_scores = {}
for r in ress:
    louvain_communities_dict, louvain_graphs = find_networks_louvain(df[resid_columns], src_subject_ids, res=r, s=42)
    s_scores = []
    for threshold in thresholds:
        comms = louvain_communities_dict[threshold]
        if len(comms) < 2:
            print(f"Threshold {threshold:.3f}: No communities found.")
            continue
        # make new df with just src_subject_id and the residualized FC columns
        df_temp = df[['src_subject_id'] + resid_columns].copy()
        df_temp.set_index('src_subject_id', inplace=True)
        score = silhouette_im(df_temp, comms, src_subject_ids)
        s_scores.append(score)
        print(f"Threshold {threshold:.3f}: Silhouette score = {score:.4f}")
# Find the resolution corresponding to the maximum silhouette score
    max_silhouette = max(s_scores)
    best_threshold_index = s_scores.index(max_silhouette)
    best_threshold = thresholds[best_threshold_index]
    print(f"Best threshold for resolution {r}: {best_threshold:.3f} with silhouette score {max_silhouette:.4f}")
    max_silhouette_scores[r] = (best_threshold, max_silhouette)

# find the optimal resolution based on the maximum silhouette score
optimal_resolution = max(max_silhouette_scores, key=lambda k: max_silhouette_scores[k][1])
optimal_threshold, optimal_silhouette = max_silhouette_scores[optimal_resolution]
print(f"Optimal resolution: {optimal_resolution}, Threshold: {optimal_threshold:.3f}, Silhouette Score: {optimal_silhouette:.4f}")



In [ ]:
# find the optimal threshold for the optimal resolution
thresholds = [i/100 for i in range(1, 51)]  # 0.01 to 0.50 in steps of 0.01
print(df[resid_columns].shape)
louvain_communities_dict, louvain_graphs = find_networks_louvain(df[resid_columns], src_subject_ids, thresholds, res=optimal_resolution, s=42)


In [ ]:
# write new function to add louvain community assignments to the DataFrame
def add_louvain_community_assignments(df, community_dict, i=1, subtypes=2):
    """
    Add Louvain community assignments to the DataFrame.

    Parameters:
    - df: DataFrame with 'src_subject_id' column
    - community_dict: dict {community_id: list of src_subject_ids}

    Returns:
    - df with new column 'louvain_community'
    """
    # Create a mapping from subject ID to community ID
    subject_to_community = {}
    for community_id, subjects in community_dict.items():
        for subject in subjects:
            subject_to_community[subject] = community_id

    # Map the communities to the DataFrame
    if subtypes == 2:
        if i == 1:
            df['louvain_community'] = df['src_subject_id'].map(subject_to_community)
        else:
            df[f'louvain_community{i}'] = df['src_subject_id'].map(subject_to_community)
    else:
        df[f'louvain_community_{subtypes}_subtypes'] = df['src_subject_id'].map(subject_to_community)
    return df

# get silhouette scores for louvain communities at each threshold
def silhouette_louvain(data, community_dict):
    """
    Compute silhouette score for Louvain communities.

    Parameters:
    - data: DataFrame of shape (n_subjects, n_features), indexed by subject ID
    - community_dict: dict {community_id: list of subject_ids}

    Returns:
    - silhouette score
    """
    from sklearn.metrics import silhouette_score
    from sklearn.metrics.pairwise import pairwise_distances

    # Build label vector from community_dict
    subject_to_label = {
        subj: label for label, subjects in community_dict.items() for subj in subjects
    }

    # Extract data in correct order
    labels = []
    valid_subjects = []
    for subj in data.index:
        if subj in subject_to_label:
            labels.append(subject_to_label[subj])
            valid_subjects.append(subj)

    if len(set(labels)) < 2:
        print("Only one community found, silhouette score cannot be calculated.")
        return -1

    X = data.loc[valid_subjects].values
    distance_matrix = pairwise_distances(X, metric='correlation')
    
    score = silhouette_score(distance_matrix, labels, metric='precomputed')
    
    return score

from sklearn.metrics import davies_bouldin_score

def davies_bouldin_louvain(data, community_dict):
    """
    Compute Davies-Bouldin score for Louvain communities.

    Parameters:
    - data: DataFrame of shape (n_subjects, n_features), indexed by subject ID
    - community_dict: dict {community_id: list of subject_ids}

    Returns:
    - Davies-Bouldin score (float) or -1 if invalid
    """
    # Build label vector from community_dict
    subject_to_label = {
        subj: label for label, subjects in community_dict.items() for subj in subjects
    }

    # Extract data and labels for subjects in the community dict
    labels = []
    valid_subjects = []
    for subj in data.index:
        if subj in subject_to_label:
            labels.append(subject_to_label[subj])
            valid_subjects.append(subj)

    from collections import Counter
    label_counts = Counter(labels)

    X = data.loc[valid_subjects].values
    db_score = davies_bouldin_score(X, labels)
    return db_score


In [ ]:
louvain_s_scores = []
for threshold in thresholds:
    comms = louvain_communities_dict[threshold]
    if len(comms) < 2:
        print(f"Threshold {threshold:.3f}: No communities found.")
        louvain_s_scores.append(-1)
        continue
    # make new df with just src_subject_id and the residualized FC columns
    df_temp = df[['src_subject_id'] + resid_columns].copy()
    df_temp.set_index('src_subject_id', inplace=True)
    score = silhouette_louvain(df_temp, comms)
    louvain_s_scores.append(score)
    print(f"Threshold {threshold:.3f}: Silhouette score = {score:.4f}")
# plot silhouette scores for louvain
plt.figure(figsize=(10, 6))
plt.plot(thresholds, [s for s in louvain_s_scores], marker='o', color='black')
plt.xlabel('Threshold')
plt.ylabel('Silhouette Score')
plt.title('Louvain Silhouette Score by Threshold')
plt.grid()
plt.show()

louvain_db_scores = []
for threshold in thresholds:
    comms = louvain_communities_dict[threshold]
    if len(comms) < 2:
        print(f"Threshold {threshold:.3f}: No communities found.")
        louvain_db_scores.append(-1)
        continue
    df_temp = df[['src_subject_id'] + resid_columns].copy()
    df_temp.set_index('src_subject_id', inplace=True)
    score = davies_bouldin_louvain(df_temp, comms)
    louvain_db_scores.append(score)
    print(f"Threshold {threshold:.3f}: Davies-Bouldin index = {score:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(thresholds, louvain_db_scores, marker='o', color='black')
plt.xlabel('Threshold')
plt.ylabel('Davies-Bouldin Index')
plt.title('Louvain Davies-Bouldin Index by Threshold')
plt.grid()
plt.show()


In [ ]:
# find threshold with lowest Davies-Bouldin score at silhouette score > 0.2
best_threshold = None
best_db_score = float('inf')
for threshold, db_score in zip(thresholds, louvain_db_scores):
    if db_score < best_db_score and db_score != -1:
        # Check if silhouette score is above 0.2
        silhouette_score = louvain_s_scores[thresholds.index(threshold)]
        if silhouette_score > 0.20:
            best_threshold = threshold
            best_db_score = db_score
print(f"Best threshold with silhouette score > 0.20: {best_threshold:.3f} with Davies-Bouldin score {best_db_score:.4f} and silhouette score {louvain_s_scores[thresholds.index(best_threshold)]:.4f}")
print(f"Associated Pearson correlation cutoff: {subject_edge_cutoff(df[resid_columns], best_threshold):.4f}")


In [ ]:
# Run Louvain repeatedly with the optimal resolution and threshold.
j = 0
for j in range(N_CLUSTERING_RUNS):
    louvain_communities_dict, louvain_graphs = find_networks_louvain(df[resid_columns], src_subject_ids, thresholds=[best_threshold], res=optimal_resolution, s=j)
    print(f"Run {j+1}/{N_CLUSTERING_RUNS}: Found {len(louvain_communities_dict[best_threshold])} communities.")
    add_louvain_community_assignments(df, louvain_communities_dict[best_threshold], i=j+1)


In [ ]:
from scipy.stats import mode
louvain_cols = [col for col in df.columns if col.startswith('louvain_community')]
df['louvain_consensus'] = mode(df[louvain_cols], axis=1).mode.flatten()
print(df['louvain_consensus'].value_counts())


In [ ]:
radar_plot(df, 'louvain_consensus')


In [ ]:
# also get most stable 3 and 4 community solutions for louvain using the default resolution of 1.0
# find the optimal threshold for 3 and 4 communities
thresholds = [i/100 for i in range(1, 51)]  # 0.01 to 0.50 in steps of 0.01
louvain_communities_dict, louvain_graphs = find_networks_louvain(df[resid_columns], src_subject_ids, thresholds, res=1.0, s=42)

# get silhouette and db scores for each threshold
louvain_s_scores = []
louvain_db_scores = []
for threshold in thresholds:
    comms = louvain_communities_dict[threshold]
    if len(comms) < 2:
        print(f"Threshold {threshold:.3f}: No communities found.")
        louvain_s_scores.append(-1)
        louvain_db_scores.append(-1)
        continue
    # make new df with just src_subject_id and the residualized FC columns
    df_temp = df[['src_subject_id'] + resid_columns].copy()
    df_temp.set_index('src_subject_id', inplace=True)
    s_score = silhouette_louvain(df_temp, comms)
    d_score = davies_bouldin_louvain(df_temp, comms)
    louvain_s_scores.append(s_score)
    louvain_db_scores.append(d_score)
    print(f"Threshold {threshold:.3f}: Silhouette score = {s_score:.4f}, Davies-Bouldin score = {d_score:.4f}")

# find best threshold for 3 communities
valid_indices_3 = np.array([i for i, s in enumerate(louvain_s_scores) if s is not None and s > 0.15 and len(louvain_communities_dict[thresholds[i]]) == 3])
if valid_indices_3.size == 0:
    print("No valid thresholds found with silhouette score > 0.15 and 3 communities")
else:
    min_db_index_3 = valid_indices_3[np.argmin(np.array(louvain_db_scores)[valid_indices_3])]
    best_threshold_3 = thresholds[min_db_index_3]
    print(f"Best threshold with silhouette score > 0.15 and 3 communities: {best_threshold_3:.3f}")
    print(f"Associated Pearson correlation cutoff: {subject_edge_cutoff(df[resid_columns], best_threshold_3):.4f}")
    print(f"Minimum Davies-Bouldin index at this threshold: {louvain_db_scores[min_db_index_3]:.4f}")
    print(f"Silhouette score at this threshold: {louvain_s_scores[min_db_index_3]:.4f}")
    community_dict_3 = louvain_communities_dict[best_threshold_3]
    df = add_louvain_community_assignments(df, community_dict_3, subtypes=3)

    radar_plot(df, 'louvain_community_3_subtypes')

# find best threshold for 4 communities
valid_indices_4 = np.array([i for i, s in enumerate(louvain_s_scores) if s is not None and s > 0.15 and len(louvain_communities_dict[thresholds[i]]) == 4])
if valid_indices_4.size == 0:
    print("No valid thresholds found with silhouette score > 0.15 and 4 communities")
else:
    min_db_index_4 = valid_indices_4[np.argmin(np.array(louvain_db_scores)[valid_indices_4])]
    best_threshold_4 = thresholds[min_db_index_4]
    print(f"Best threshold with silhouette score > 0.15 and 4 communities: {best_threshold_4:.3f}")
    print(f"Associated Pearson correlation cutoff: {subject_edge_cutoff(df[resid_columns], best_threshold_4):.4f}")
    print(f"Minimum Davies-Bouldin index at this threshold: {louvain_db_scores[min_db_index_4]:.4f}")
    print(f"Silhouette score at this threshold: {louvain_s_scores[min_db_index_4]:.4f}")
    community_dict_4 = louvain_communities_dict[best_threshold_4]
    df = add_louvain_community_assignments(df, community_dict_4, subtypes=4)
    radar_plot(df, 'louvain_community_4_subtypes')


In [ ]:
import datetime
now = datetime.datetime.now()
date_str = now.strftime('%Y-%m-%d')
hour_str = now.strftime('%H-%M')

output_filename = f'midb61_meanFC_clusters_{CLUSTERING_OUTPUT_LABEL}_{date_str}_{hour_str}.csv'
df.to_csv(os.path.join(OUTPUT_DIR, output_filename), index=False)
print('Saved clustering output to:', output_filename)


## FC-PCA Visualization and Clustering Stability

Use the FC-PCA scores exported by `PCA_FC.ipynb` to visualize the two- and four-cluster solutions. Assess sample stability by rerunning K-Means and Louvain on 5,000 with-replacement subject bootstrap samples and calculating cluster-wise Jaccard recovery. Rand Index is calculated separately across the 1,001 full-sample runs and between final algorithms.

In [ ]:
# Plot existing cluster assignments in the PCA space fitted by PCA_FC.ipynb.
# This cell loads saved scores; it does not fit or transform a new PCA.
pca_model_name = (
    'with_ses_residualization'
    if RESIDUALIZE_FC_FOR_INR
    else 'without_ses_residualization'
)
pca_prefix = 'ses_fc' if RESIDUALIZE_FC_FOR_INR else 'no_ses_fc'
pca_results_dir = os.path.join(OUTPUT_DIR, 'pca_results')
pca_score_path = os.path.join(
    pca_results_dir,
    f'pca_fc_scores_{pca_model_name}.csv',
)
pca_variance_path = os.path.join(
    pca_results_dir,
    f'pca_fc_variance_{pca_model_name}.csv',
)


pca_scores = pd.read_csv(pca_score_path)
pca_scores['src_subject_id'] = standardize_subject_id(pca_scores['src_subject_id'])
pca_pc1 = f'{pca_prefix}_PC1'
pca_pc2 = f'{pca_prefix}_PC2'

pca_scores = pca_scores[['src_subject_id', pca_pc1, pca_pc2]].drop_duplicates(
    subset='src_subject_id'
)
df = df.drop(columns=[pca_pc1, pca_pc2], errors='ignore').merge(
    pca_scores,
    on='src_subject_id',
    how='left',
)

pca_variance = pd.read_csv(pca_variance_path)
pc1_variance = 100 * pca_variance.loc[
    pca_variance['component'].eq(1), 'explained_variance_ratio'
].iloc[0]
pc2_variance = 100 * pca_variance.loc[
    pca_variance['component'].eq(2), 'explained_variance_ratio'
].iloc[0]

pca_solutions = [
    ('K-Means', 2, 'kmeans_2_consensus'),
    ('Louvain', 2, 'louvain_consensus'),
    ('K-Means', 4, 'kmeans_4_labels'),
    ('Louvain', 4, 'louvain_community_4_subtypes'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 11), sharex=True, sharey=True)
for ax, (algorithm, n_clusters, label_column) in zip(axes.flat, pca_solutions):
    plot_data = df[[pca_pc1, pca_pc2, label_column]].dropna().copy()
    plot_data['Community'] = plot_data[label_column].astype(int) + 1
    communities = sorted(plot_data['Community'].unique())
    palette_map = dict(zip(
        communities,
        sns.color_palette('colorblind', n_colors=len(communities)),
    ))

    sns.scatterplot(
        data=plot_data,
        x=pca_pc1,
        y=pca_pc2,
        hue='Community',
        palette=palette_map,
        s=24,
        alpha=0.60,
        linewidth=0,
        ax=ax,
    )

    centers = plot_data.groupby('Community')[[pca_pc1, pca_pc2]].mean()
    for community, center in centers.iterrows():
        ax.scatter(
            center[pca_pc1],
            center[pca_pc2],
            marker='X',
            s=140,
            color=palette_map[community],
            edgecolor='black',
            linewidth=0.8,
            zorder=5,
        )

    ax.set_title(f'{algorithm}: {n_clusters}-cluster solution')
    ax.set_xlabel(f'FC PC1 ({pc1_variance:.1f}% variance)')
    ax.set_ylabel(f'FC PC2 ({pc2_variance:.1f}% variance)')
    ax.legend(title='Community', frameon=True, markerscale=1.3)

fig.suptitle(
    f'Cluster solutions in established FC-PCA space: {CLUSTERING_OUTPUT_LABEL}',
    fontsize=15,
    y=1.01,
)
fig.tight_layout()

pca_figure_filename = (
    f'cluster_solutions_existing_FC_PCA_{CLUSTERING_OUTPUT_LABEL}_'
    f'{date_str}_{hour_str}.png'
)
fig.savefig(
    os.path.join(OUTPUT_DIR, pca_figure_filename),
    dpi=300,
    bbox_inches='tight',
)
plt.show()
print('Saved FC-PCA cluster visualization to:', pca_figure_filename)


### 1. Agreement across repeated full-sample runs

Calculate mean pairwise Rand Index across the 1,001 runs and compare the final K-Means and Louvain assignments.


In [ ]:
from itertools import combinations
from sklearn.metrics import rand_score


kmeans_run_columns = [
    f'kmeans_2_run_{run}' for run in range(1, N_CLUSTERING_RUNS + 1)
]
louvain_run_columns = [
    'louvain_community',
    *[f'louvain_community{run}' for run in range(2, N_CLUSTERING_RUNS + 1)],
]

kmeans_mean_rand = np.mean([
    rand_score(df[left], df[right])
    for left, right in combinations(kmeans_run_columns, 2)
])
louvain_mean_rand = np.mean([
    rand_score(df[left], df[right])
    for left, right in combinations(louvain_run_columns, 2)
])

rand_results = pd.DataFrame([
    {
        'comparison': 'Mean pairwise Rand Index across K-Means runs',
        'clusters': 2,
        'rand_index': kmeans_mean_rand,
        'n_runs': N_CLUSTERING_RUNS,
    },
    {
        'comparison': 'Mean pairwise Rand Index across Louvain runs',
        'clusters': 2,
        'rand_index': louvain_mean_rand,
        'n_runs': N_CLUSTERING_RUNS,
    },
    {
        'comparison': 'Final K-Means versus Louvain Rand Index',
        'clusters': 2,
        'rand_index': rand_score(df['kmeans_2_consensus'], df['louvain_consensus']),
        'n_runs': np.nan,
    },
    {
        'comparison': 'Final K-Means versus Louvain Rand Index',
        'clusters': 4,
        'rand_index': rand_score(df['kmeans_4_labels'], df['louvain_community_4_subtypes']),
        'n_runs': np.nan,
    },
])

display(rand_results)
rand_filename = f'clustering_rand_indices_{CLUSTERING_OUTPUT_LABEL}_{date_str}_{hour_str}.csv'
rand_results.to_csv(os.path.join(OUTPUT_DIR, rand_filename), index=False)
print('Saved Rand Index results to:', rand_filename)


### 2. Prepare the bootstrap analysis

Use the residualized FC profiles and the final two- and four-cluster assignments as references.


In [ ]:
reference_columns = {
    ('K-Means', 2): 'kmeans_2_consensus',
    ('K-Means', 4): 'kmeans_4_labels',
    ('Louvain', 2): 'louvain_consensus',
    ('Louvain', 4): 'louvain_community_4_subtypes',
}

bootstrap_complete = df[[*resid_columns, *reference_columns.values()]].notna().all(axis=1)
bootstrap_features = df.loc[bootstrap_complete, resid_columns].to_numpy(dtype=float)
reference_labels = {
    key: df.loc[bootstrap_complete, column].to_numpy()
    for key, column in reference_columns.items()
}
print(f'Subjects used for bootstrap stability: {bootstrap_complete.sum()}/{len(df)}')


def fit_louvain(similarity_matrix, threshold, resolution, random_state):
    upper_i, upper_j = np.triu_indices(len(similarity_matrix), k=1)
    upper_values = similarity_matrix[upper_i, upper_j]
    cutoff = np.percentile(upper_values, 100 - threshold * 100)
    retained = upper_values > cutoff

    graph = nx.Graph()
    graph.add_nodes_from(range(len(similarity_matrix)))
    graph.add_weighted_edges_from(
        (int(i), int(j), float(weight))
        for i, j, weight in zip(
            upper_i[retained], upper_j[retained], upper_values[retained]
        )
    )
    communities = louvain_communities(
        graph, resolution=resolution, seed=random_state, weight='weight'
    )
    labels = np.empty(len(similarity_matrix), dtype=int)
    for community, members in enumerate(communities):
        labels[list(members)] = community
    return labels


def fit_bootstrap_solutions(sample_indices, random_state):
    sample = bootstrap_features[sample_indices]
    similarity = np.corrcoef(sample)
    return {
        ('K-Means', 2): KMeans(n_clusters=2, random_state=random_state).fit_predict(sample),
        ('K-Means', 4): KMeans(n_clusters=4, random_state=random_state).fit_predict(sample),
        ('Louvain', 2): fit_louvain(
            similarity, best_threshold, optimal_resolution, random_state
        ),
        ('Louvain', 4): fit_louvain(
            similarity, best_threshold_4, 1.0, random_state
        ),
    }


### 3. Run 5,000 bootstrap resamples

Each sample draws the same number of subjects with replacement, refits K-Means and Louvain, and records best-matched cluster Jaccard recovery.


In [ ]:
bootstrap_rng = np.random.default_rng(CLUSTER_BOOTSTRAP_RANDOM_STATE)
bootstrap_records = []

for bootstrap_index in range(N_CLUSTER_BOOTSTRAPS):
    # Draw n subjects with replacement.
    sample_indices = bootstrap_rng.integers(
        0, len(bootstrap_features), size=len(bootstrap_features)
    )
    fitted_solutions = fit_bootstrap_solutions(
        sample_indices,
        random_state=CLUSTER_BOOTSTRAP_RANDOM_STATE + bootstrap_index,
    )

    for (algorithm, n_clusters), fitted_labels in fitted_solutions.items():
        # A subject may be drawn more than once. Use its modal fitted label.
        collapsed = (
            pd.DataFrame({'subject_index': sample_indices, 'label': fitted_labels})
            .groupby('subject_index', as_index=False)['label']
            .agg(lambda values: values.value_counts().index[0])
            .sort_values('subject_index')
        )
        unique_indices = collapsed['subject_index'].to_numpy(dtype=int)
        candidate_labels = collapsed['label'].to_numpy()
        reference = reference_labels[(algorithm, n_clusters)][unique_indices]

        # Match each reference cluster to the bootstrap cluster with greatest Jaccard overlap.
        for reference_cluster in np.unique(reference):
            reference_members = reference == reference_cluster
            jaccard = max(
                np.logical_and(reference_members, candidate_labels == candidate_cluster).sum()
                / np.logical_or(reference_members, candidate_labels == candidate_cluster).sum()
                for candidate_cluster in np.unique(candidate_labels)
            )
            bootstrap_records.append({
                'bootstrap': bootstrap_index + 1,
                'algorithm': algorithm,
                'clusters': n_clusters,
                'reference_cluster': int(reference_cluster),
                'jaccard': jaccard,
                'n_unique_in_bag': len(unique_indices),
                'n_recovered_clusters': len(np.unique(candidate_labels)),
            })

    if (bootstrap_index + 1) % 50 == 0:
        print(f'Completed {bootstrap_index + 1}/{N_CLUSTER_BOOTSTRAPS} resamples')


### 4. Summarize cluster recovery


In [ ]:
bootstrap_jaccard = pd.DataFrame(bootstrap_records)
bootstrap_summary = (
    bootstrap_jaccard
    .groupby(['algorithm', 'clusters', 'reference_cluster'])
    .agg(
        mean_jaccard=('jaccard', 'mean'),
        median_jaccard=('jaccard', 'median'),
        sd_jaccard=('jaccard', 'std'),
        ci_95_low=('jaccard', lambda values: values.quantile(0.025)),
        ci_95_high=('jaccard', lambda values: values.quantile(0.975)),
        mean_unique_in_bag=('n_unique_in_bag', 'mean'),
        mean_recovered_clusters=('n_recovered_clusters', 'mean'),
        n_bootstrap=('bootstrap', 'nunique'),
    )
    .reset_index()
)

display(bootstrap_summary)
bootstrap_raw_filename = (
    f'clustering_bootstrap_jaccard_raw_{CLUSTERING_OUTPUT_LABEL}_{date_str}_{hour_str}.csv'
)
bootstrap_summary_filename = (
    f'clustering_bootstrap_jaccard_summary_{CLUSTERING_OUTPUT_LABEL}_{date_str}_{hour_str}.csv'
)
bootstrap_jaccard.to_csv(os.path.join(OUTPUT_DIR, bootstrap_raw_filename), index=False)
bootstrap_summary.to_csv(os.path.join(OUTPUT_DIR, bootstrap_summary_filename), index=False)
print('Saved raw bootstrap results to:', bootstrap_raw_filename)
print('Saved bootstrap summary to:', bootstrap_summary_filename)


In [ ]:
from sklearn.metrics import silhouette_score as skl_silhouette_score, davies_bouldin_score as skl_davies_bouldin_score, calinski_harabasz_score as skl_calinski_harabasz_score

# Calculate validation metrics using rows with complete FC profiles and all three clustering labels.
resid_cols = cluster_resid_columns
metric_columns = resid_cols + ['kmeans_2_consensus', 'louvain_consensus', 'infomap_community_2']
metric_complete = df[metric_columns].notna().all(axis=1)
print(f'Rows used for clustering validation metrics: {metric_complete.sum()}/{len(df)}')

X = df.loc[metric_complete, resid_cols].values
kmeans_labels = df.loc[metric_complete, 'kmeans_2_consensus'].values
dbi_kmeans = skl_davies_bouldin_score(X, kmeans_labels)
print(f'Davies-Bouldin Index for KMeans clustering: {dbi_kmeans:.4f}')

louvain_labels = df.loc[metric_complete, 'louvain_consensus'].values
dbi_louvain = skl_davies_bouldin_score(X, louvain_labels)
print(f'Davies-Bouldin Index for Louvain clustering: {dbi_louvain:.4f}')

infomap_labels = df.loc[metric_complete, 'infomap_community_2'].values
dbi_infomap = skl_davies_bouldin_score(X, infomap_labels)
print(f'Davies-Bouldin Index for Infomap clustering: {dbi_infomap:.4f}')

silhouette_kmeans_score = skl_silhouette_score(X, kmeans_labels, metric='correlation')
print(f'Silhouette Score for KMeans clustering: {silhouette_kmeans_score:.4f}')

silhouette_louvain_score = skl_silhouette_score(X, louvain_labels, metric='correlation')
print(f'Silhouette Score for Louvain clustering: {silhouette_louvain_score:.4f}')

silhouette_infomap_score = skl_silhouette_score(X, infomap_labels, metric='correlation')
print(f'Silhouette Score for Infomap clustering: {silhouette_infomap_score:.4f}')

ch_kmeans = skl_calinski_harabasz_score(X, kmeans_labels)
print(f'Calinski-Harabasz Score for KMeans clustering: {ch_kmeans:.4f}')

ch_louvain = skl_calinski_harabasz_score(X, louvain_labels)
print(f'Calinski-Harabasz Score for Louvain clustering: {ch_louvain:.4f}')

ch_infomap = skl_calinski_harabasz_score(X, infomap_labels)
print(f'Calinski-Harabasz Score for Infomap clustering: {ch_infomap:.4f}')
